# Data Exploration Primer

**Your task:** explore historical house sales and explain what the data can tell
us about prices. At the end, you will shortlist sale records for a buyer with
specific requirements.

By the end of this primer, you should be able to:

- inspect a table and distinguish measurements, categories, identifiers, and dates;
- choose and interpret simple summaries and plots;
- filter records, create a new measure, and rank a subset;
- support a finding with evidence and acknowledge a limitation.

## How to work through this notebook

Work through **Exercises 1–4** in order. The extensions at the end are optional.
Read and run the **worked examples**, then complete cells marked **Your turn**.
When a question asks for a prediction, write it **before** running the next cell.
Predictions are not graded for being correct; explain what you learned afterwards.

Write short answers in Markdown cells: two or three sentences are usually enough.
To edit a Markdown cell, double-click it; run it to display the formatted text.

A pandas **DataFrame** is a table. `houses["price"]` selects one column, and
`houses[["price", "grade"]]` selects a table containing two columns.
A method such as `.head()` performs an operation on the object before the dot.

**Before submitting:** restart the kernel and run all cells from top to bottom.
Incomplete tasks must be completed even if they do not raise an error.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

## The dataset and setup

Keep **`kc_house_data.csv` in the same folder as this notebook**. If using
Google Colab, upload the CSV into the session's files before running the next cell.
Use a Python environment with `pandas` and `matplotlib` installed.

The file contains **21,613 sale records** in **King County, Washington, USA**,
dated **2 May 2014 to 27 May 2015**. Each row is a recorded sale: the same house
identifier can occur more than once. These are historical transactions, not
current listings or a census of every house in the county.

In [ ]:
# The two paths support starting Jupyter in this folder or in the project folder.
possible_paths = [Path("kc_house_data.csv"), Path("Primer Assignments") / "kc_house_data.csv"]
data_path = next((path for path in possible_paths if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError("Put kc_house_data.csv beside this notebook, then run this cell again.")

houses = pd.read_csv(data_path)
houses.head()  # Jupyter displays the last expression in a cell.

### A small data dictionary

We will focus on the following columns; you do not need to use every column.

| Column | Meaning and units | Important detail |
|---|---|---|
| `id` | House identifier | A label, not a measurement; repeated sales can share an ID. |
| `date` | Sale date, stored as text | For example, `20141013T000000` means 13 October 2014. |
| `price` | Recorded sale price, USD | Historical price, not a current valuation. |
| `bedrooms` | Number of bedrooms | A numerical count. |
| `bathrooms` | Recorded bathroom count | Includes fractional values for partial bathrooms. |
| `floors` | Recorded number of floors | Can include fractional values. |
| `sqft_living` | Living area, square feet | Different from lot area (`sqft_lot`). |
| `zipcode` | Postal ZIP code | An unordered category, even when stored as an integer. |
| `grade` | Ordered construction/design rating | Larger values indicate a higher rating; equal steps need not mean equal quality differences. |
| `yr_renovated` | Recorded renovation year | In this teaching copy, 0 means no renovation year is recorded. |
| `lat`, `long` | Latitude and longitude, degrees | Coordinates of the property. |

`waterfront` is a 0/1 waterfront indicator, `condition` is an ordered condition
rating, and `sqft_basement` is basement area in square feet (0 means no basement
area is recorded). These columns appear in the optional extensions.

## Exercise 1 — Understand the table and its summaries

**Worked example:** `.shape` gives the number of rows and columns, `.columns`
lists their names, and `.dtypes` shows their Python storage types.
Storage type does not determine a variable's statistical meaning.

In [ ]:
print("Rows and columns:", houses.shape)
print("Column names:", houses.columns.tolist())
houses[["price", "bedrooms", "zipcode", "grade", "id", "date"]].dtypes

**Your turn — classify by meaning.** Complete the table. Use the roles
*numerical measurement*, *numerical count*, *unordered category*,
*ordered category*, *identifier*, or *date/time*.

| Column | Your classification | A useful summary, and why |
|---|---|---|
| `sqft_living` | _your answer_ | _your answer_ |
| `bedrooms` | _your answer_ | _your answer_ |
| `zipcode` | _your answer_ | _your answer_ |
| `grade` | _your answer_ | _your answer_ |
| `id` | _your answer_ | _your answer_ |
| `date` | _your answer_ | _your answer_ |

Why would the average of the ZIP codes or house identifiers be unhelpful?

**Your explanation:** _write here_

**Worked example — check missing values.** `.isna()` identifies missing entries;
`.sum()` counts them in each column. A zero missing-value count does not prove
that every value is correct or informative.

In [ ]:
houses.isna().sum()

**Worked example — numerical summaries.** `.describe()` shows the number of
non-missing values, mean, standard deviation, minimum, quartiles, and maximum.
The 50% value is the **median**: half the observations are at or below it.

Read the table below. Notice the units and the difference between a typical
value and an extreme value.

In [ ]:
houses[["price", "sqft_living", "bedrooms", "yr_renovated"]].describe().round(2)

**Your turn — investigate three summaries.** In the next code cell:

1. Calculate the mean and median price using `.mean()` and `.median()`.
2. Use `.value_counts().head()` on `yr_renovated` to find its most common values.
3. Inspect rows with an unusually high bedroom count. The example below shows
   how to select records satisfying one condition; it does not alter the data.

Then answer the questions underneath. Do not remove unusual observations
automatically: investigating them is part of exploration.

In [ ]:
# TODO: calculate and print the mean and median sale price.
# Hint: houses["price"].mean() and houses["price"].median()

# TODO: show the five most common values of yr_renovated.
# Hint: houses["yr_renovated"].value_counts().head()

# Supplied example: a Boolean condition selects records with more than 10 bedrooms.
houses.loc[houses["bedrooms"] > 10, ["id", "bedrooms", "sqft_living", "price"]]

**Mean and median:** Which price summary would you use to describe a typical
recorded sale? Give the two values and explain your choice.

_Your answer:_

**Zeros and missing values:** Why does the mean of `yr_renovated` not describe
a typical renovation year? How could you summarize this column more usefully?

_Your answer:_

**Unusual values:** Identify one value you would investigate. What information
would you need before deciding to correct or remove it?

_Your answer:_

## Exercise 2 — Explore distributions and relationships

A **histogram** groups numerical values into intervals and shows how many
observations fall in each interval. A **scatterplot** shows two values for each
record, helping us examine an association.

**Predict first:** Do you expect prices to be evenly distributed? Do you expect
houses with more living area to have higher recorded sale prices? Explain briefly.

_Your prediction:_

**Worked examples.** `fig` is the whole figure and `ax` is its plotting area.
We create a plot, add labels with units, and call `plt.show()` to display it.
In the scatterplot, `s` controls point size and `alpha` controls transparency:
small, translucent points help when many observations overlap.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(houses["price"], bins=30, color="steelblue", edgecolor="white")
ax.set_xlabel("Recorded sale price (USD)")
ax.set_ylabel("Number of sale records")
ax.set_title("Distribution of recorded sale prices")
ax.ticklabel_format(style="plain", axis="x")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(houses["sqft_living"], houses["price"], s=12, alpha=0.25)
ax.set_xlabel("Living area (square feet)")
ax.set_ylabel("Recorded sale price (USD)")
ax.set_title("Living area and sale price")
ax.ticklabel_format(style="plain", axis="y")
plt.show()

**Your turn — adapt the plotting pattern.** Choose `bathrooms` or `bedrooms`,
and make a scatterplot of that variable against price. Give it a title and
meaningful axis labels. You can copy and modify the previous example.

Make one prediction before running your plot. If the points overlap heavily,
use a smaller `s` or `alpha`; explain why an empty-looking region or a dense
cluster might be difficult to interpret.

_Your chosen variable and prediction:_

In [ ]:
x_column = "bathrooms"  # You may choose "bedrooms" instead.

# TODO: make a scatterplot of houses[x_column] against houses["price"].
# Include a title, axis labels with units, and plt.show().

**Distribution:** How does the price histogram help explain the difference
between the mean and median from Exercise 1?

_Your answer:_

**Relationship:** Describe the direction and spread in your chosen scatterplot.
Does the trend describe every sale? Support your answer with something visible
in the plot.

_Your answer:_

**Limits:** A larger house may also differ in location, condition, or grade.
Why does an association with price not establish that changing one feature
would cause the price to change? Suggest one other feature to consider.

_Your answer:_

## Exercise 3 — Compare prices across ordered categories

`grade` is an ordered rating. First examine **how many sales are in each grade**;
comparing a group with very few records to a large group requires care.

**Worked example:** `.value_counts()` counts records in each category and
`.sort_index()` puts grades in their numerical order. Here a bar chart is
appropriate because each bar represents a distinct category.

In [ ]:
grade_counts = houses["grade"].value_counts().sort_index()
print(grade_counts)

fig, ax = plt.subplots(figsize=(7, 4))
grade_counts.plot.bar(ax=ax, color="steelblue", rot=0)
ax.set_xlabel("Construction/design grade")
ax.set_ylabel("Number of sale records")
ax.set_title("Number of recorded sales by grade")
plt.show()

**Your turn:** use the scatterplot pattern to plot `grade` on the horizontal
axis and `price` on the vertical axis. This will give a useful comparison
with the boxplot below.

In [ ]:
# TODO: plot grade against price, including readable labels and a title.
# Hint: adapt the worked scatterplot from Exercise 2.

**Worked example — a boxplot for each grade.** A boxplot summarizes a numerical
distribution. The line inside a box is the **median**; the bottom and top are
the 25th and 75th percentiles, so the box contains the middle 50% of observations.
The box height is the **interquartile range (IQR)**.

In this plot, whiskers extend to the most extreme observations still within
1.5 IQR of the box. Points beyond the whiskers are flagged as outliers.
They are not automatically errors or evidence that a house was overpriced.

pandas creates the groups and labels them with their actual grade values.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
houses.boxplot(column="price", by="grade", ax=ax, grid=False)
fig.suptitle("")  # Remove the extra title that pandas adds automatically.
ax.set_title("Recorded sale price distributions by grade")
ax.set_xlabel("Construction/design grade")
ax.set_ylabel("Recorded sale price (USD)")
ax.ticklabel_format(style="plain", axis="y")
plt.show()

**Compare:** What does the boxplot make easier to see than your grade–price
scatterplot? Describe a pattern in the medians and a pattern in the spread.

_Your answer:_

**Check the group sizes:** Name a grade with few records. Why should a claim
about its price distribution be cautious?

_Your answer:_

**Critique:** A student says, "All points above the whiskers are overpriced
houses." Explain why the plot does not support that conclusion.

_Your answer:_

## Exercise 4 — Shortlist sales that meet a buyer's requirements

A buyer wants a house with **exactly 3 bedrooms, exactly 2 bathrooms, and exactly
2 floors**. Use historical sales to explore possible trade-offs; these houses
are not necessarily for sale today.

**Worked example — filtering:** a Boolean mask contains `True` for records
that satisfy a condition. `.loc[mask, columns]` selects those rows and columns.
When combining conditions, put each in parentheses and use `&` for **and**
(`|` means **or**). `.copy()` creates a separate table for the subset.

In [ ]:
large_mask = houses["sqft_living"] >= 2000
large_sales = houses.loc[large_mask, ["id", "price", "sqft_living"]].copy()
print("Sale records with at least 2,000 square feet:", len(large_sales))
large_sales.head()

**Your turn — combine three conditions.** Create `matching_houses`, keeping
**all columns**, using the buyer's exact requirements. Start from `houses`,
combine the three conditions, and use `.copy()`.

Replace `None` below with your selection. `None` is a placeholder, not an answer.
Then run the supplied checkpoint. It should confirm that the selected records
satisfy all three requirements.

In [ ]:
matching_houses = None  # TODO: replace None with your filtered DataFrame.

# Hint for one condition: (houses["bedrooms"] == 3)
# Join conditions with & and select rows using houses.loc[...].copy().

In [ ]:
# Supplied checkpoint: run this after completing your filter.
if matching_houses is None:
    print("INCOMPLETE: create matching_houses in the previous task cell.")
else:
    assert len(matching_houses) > 0, "Your selection is empty; check the conditions."
    assert (matching_houses["bedrooms"] == 3).all(), "Check the bedroom condition."
    assert (matching_houses["bathrooms"] == 2).all(), "Check the bathroom condition."
    assert (matching_houses["floors"] == 2).all(), "Check the floor condition."
    print("Checkpoint passed: every selected record satisfies the requirements.")
    print("Number of matching sale records:", len(matching_houses))

**Your turn — summarize the selection.** Report the number of matching sales
and use `.describe()` on their `price` and `sqft_living` columns. Compare the
subset's median price with the median for the full dataset from Exercise 1.

In [ ]:
# TODO: summarize matching_houses once you have completed the filter.
# Hint: matching_houses[["price", "sqft_living"]].describe()

**Your turn — create a measure and rank the subset.**

Define **price per square foot = price / living area**, measured in USD per
square foot. For example, a sale for $300,000 with 2,000 square feet has a value
of $150 per square foot. **Smaller values mean a lower price per unit of space.**

1. Add a column called `price_per_sqft` to `matching_houses` using column-wise
   division: the operation calculates one ratio for each row.
2. Use `.sort_values("price_per_sqft")` and `.head(5)` to create `shortlist`.
   **Keep all columns**, so the checkpoint can verify your selection.
3. Show a readable table containing `id`, `date`, `price`, `sqft_living`,
   `price_per_sqft`, and `grade`. An ID and date together identify the sale.

Continue working with `matching_houses` when ranking: the buyer's requirements
still apply. Sorting by the ratio changes the order, not the requirements.

In [ ]:
# TODO: add matching_houses["price_per_sqft"] using price / sqft_living.

shortlist = None  # TODO: replace None with the five lowest-ratio matching rows.

# TODO: display the requested columns from shortlist.

In [ ]:
# Supplied checkpoint. It checks constraints, units, and ordering, not your explanation.
if matching_houses is None or shortlist is None:
    print("INCOMPLETE: finish both the filtering and ranking tasks.")
else:
    assert len(shortlist) == min(5, len(matching_houses)), "Check the shortlist length."
    assert (shortlist["bedrooms"] == 3).all(), "A shortlisted sale fails the bedroom requirement."
    assert (shortlist["bathrooms"] == 2).all(), "A shortlisted sale fails the bathroom requirement."
    assert (shortlist["floors"] == 2).all(), "A shortlisted sale fails the floor requirement."
    assert (shortlist["sqft_living"] > 0).all(), "Check the living areas before dividing."
    expected_ratios = shortlist["price"] / shortlist["sqft_living"]
    assert ((shortlist["price_per_sqft"] - expected_ratios).abs() < 0.000001).all(), "Check the ratio's numerator and denominator."
    assert shortlist["price_per_sqft"].is_monotonic_increasing, "Sort from lowest to highest ratio."
    assert shortlist.index.is_unique and shortlist.index.isin(matching_houses.index).all(), "Keep the original sale-row indexes when filtering and sorting."
    cutoff = (matching_houses["price"] / matching_houses["sqft_living"]).nsmallest(len(shortlist)).max()
    assert (shortlist["price_per_sqft"] <= cutoff + 0.000001).all(), "Some selected records have a lower ratio than a shortlisted record."
    print("Checkpoint passed: the shortlist contains the lowest ratios among matching sales.")

**Worked example — highlight a subset.** Once you have created the shortlist,
run the following cell. The grey points show all sales, the blue points show
matching sales, and the orange outlines show shortlisted sales.
All points use the same axes, allowing a direct comparison.

In [ ]:
if matching_houses is None or shortlist is None:
    print("INCOMPLETE: finish the selection and shortlist to display this plot.")
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(houses["sqft_living"], houses["price"], color="grey", s=10, alpha=0.15, label="All sales")
    ax.scatter(matching_houses["sqft_living"], matching_houses["price"], color="steelblue", s=25, alpha=0.6, label="Matching sales")
    ax.scatter(shortlist["sqft_living"], shortlist["price"], facecolors="none", edgecolors="darkorange", s=110, linewidths=2, label="Shortlist")
    ax.set_xlabel("Living area (square feet)")
    ax.set_ylabel("Recorded sale price (USD)")
    ax.set_title("Matching sales and the price-per-square-foot shortlist")
    ax.ticklabel_format(style="plain", axis="y")
    ax.legend()
    plt.show()

**Summarize:** How many sales match the requirements? How does their median
price compare with the full dataset's median?

_Your answer:_

**Recommend:** Choose one shortlisted sale and identify it by ID and date.
Give its price, living area, and price per square foot. Explain one advantage
and one trade-off, using evidence from your table or plot.

_Your answer:_

**Limit the conclusion:** Does the lowest price per square foot necessarily
identify the best house for this buyer? Mention a factor the ratio ignores,
and explain why a historical sale price cannot be treated as a current offer.

_Your answer:_

## Final reflection and submission

**Two findings:** Give two findings supported by your analysis. Refer to a
numerical result or a specific feature of a plot for each.

_Your answer:_

**One limitation:** State one limitation affecting what you can conclude from
these sale records. Remember that recorded sales are not all homes in the county.

_Your answer:_

**One next question:** Propose a small question you could investigate in an
advanced project. State which variables or additional data you would need.

_Your answer:_

### Submission checklist

- Complete all core TODOs and written answers; optional extensions are not required.
- Give plots readable titles and labels with units where applicable.
- Run both Exercise 4 checkpoints successfully.
- Restart the kernel and run all cells; save the notebook with your results.
- Submit this `.ipynb` file following your instructor's naming instructions.

Your work should demonstrate correct analysis, explanations supported by evidence,
and clear communication of limitations. A technically elaborate plot is not required.

## Optional extensions — choose one if you have time

### A. Critique a ZIP-code scatterplot

The following plot deliberately places integer ZIP codes on a numerical axis.
ZIP codes are labels: subtracting two codes does not measure geographical distance.
Inspect the plot, then explain why a weak linear trend would **not** show that
location is unrelated to price.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(houses["zipcode"], houses["price"], s=10, alpha=0.2)
ax.set_xlabel("ZIP code (a category displayed as a number)")
ax.set_ylabel("Recorded sale price (USD)")
ax.set_title("A plot to critique: numeric ZIP code versus price")
ax.ticklabel_format(style="plain", axis="both", useOffset=False)
plt.show()

_Your critique:_

**Try an alternative:** select three ZIP codes with many recorded sales and
compare their price distributions using `subset.boxplot(column="price", by="zipcode")`.
A selection pattern is `houses.loc[houses["zipcode"].isin(chosen_codes)].copy()`,
where `chosen_codes` is your list of three codes. Report the group sizes.

_Your comparison:_

In [ ]:
# Optional TODO: choose three ZIP codes, create a subset, and compare distributions.
# Hint: houses["zipcode"].value_counts().head() helps find well-represented groups.

### B. Explore locations using coordinates

Run this worked example. Longitude is on the horizontal axis, latitude on the
vertical axis, and colour represents grade. This is a coordinate plot, not a
street map; numerical steps in latitude and longitude are not equal distances.

Describe a visible spatial pattern, then state a limitation. Dense clusters
indicate more **recorded sales**, and do not by themselves identify a city centre
or prove that most houses in an area have a particular grade.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
points = ax.scatter(houses["long"], houses["lat"], c=houses["grade"], cmap="viridis", s=10, alpha=0.5)
fig.colorbar(points, ax=ax, label="Construction/design grade")
ax.set_xlabel("Longitude (degrees)")
ax.set_ylabel("Latitude (degrees)")
ax.set_title("Locations of recorded house sales, coloured by grade")
plt.show()

_Your observation and limitation:_

### C. Compare two categorical variables

Start with a **contingency table**: each cell counts records with one combination
of waterfront status and condition rating. The second table shows proportions
**within each waterfront group**, so each row sums to 1. Always identify the
denominator when reporting a proportion; a group count is not a population risk.

A **mosaic plot** represents combinations with rectangles whose areas correspond
to their counts. It needs the optional `statsmodels` package. The plotting lines
below are commented out so this extension adds no required dependency. If your
instructor's environment includes the package, uncomment and run them.

In [ ]:
category_counts = pd.crosstab(houses["waterfront"], houses["condition"])
print("Counts (waterfront: 0 = no, 1 = yes):")
print(category_counts)
print("Proportions within each waterfront group:")
print(pd.crosstab(houses["waterfront"], houses["condition"], normalize="index").round(3))

# Optional mosaic plot, if statsmodels is installed:
# from statsmodels.graphics.mosaicplot import mosaic
# fig, rectangles = mosaic(houses, ["waterfront", "condition"])
# fig.set_size_inches(8, 5)
# fig.axes[0].set_title("Recorded sales by waterfront status and condition")
# plt.show()

Which combination has the most recorded sales? Compare the proportion of
condition-5 sales within each waterfront group. Report both group sizes and
explain why comparing counts alone could be misleading.

_Your answer:_